In [ ]:
import numpy as np

# 1. Define the Objective Function (Fitness Function)
def fitness_function(chromosome):
    """Calculates the fitness (value) of the objective function f(x) = x^2 + 5."""

    # Convert binary chromosome (list of 0s and 1s) to its decimal value (x)
    x = binary_to_decimal(chromosome)

    # Calculate the function value (fitness)
    fitness = (x**2) + 5
    return fitness

def binary_to_decimal(chromosome):
    """Converts a binary list/array to a decimal integer."""
    # Chromosome is a list/array of 0s and 1s (e.g., [1, 0, 1, 0, 0] = 20)
    decimal_value = 0
    for i, bit in enumerate(chromosome[::-1]):  # Iterate in reverse (LSB first)
        decimal_value += bit * (2 ** i)
    return decimal_value

# 2. GA Components
def initialize_population(pop_size, gene_length):
    """Creates a random initial population of chromosomes."""
    return np.random.randint(0, 2, size=(pop_size, gene_length))

def selection(population, fitnesses, num_parents):
    """Selects parents using Roulette Wheel Selection (proportional to fitness)."""

    # Normalize fitnesses to get probabilities
    probabilities = fitnesses / np.sum(fitnesses)

    # Select indices based on probabilities
    parent_indices = np.random.choice(
        np.arange(len(population)),
        size=num_parents,
        p=probabilities,
        replace=False # Ensures unique parents for crossover
    )
    return population[parent_indices]

def crossover(parents, offspring_size, crossover_rate=0.7):
    """Performs single-point crossover to generate offspring."""
    offspring = np.empty(offspring_size, dtype=int)

    for k in range(offspring_size[0]):
        # Select two parents
        parent1 = parents[k % parents.shape[0]]
        parent2 = parents[(k + 1) % parents.shape[0]]

        # Determine crossover point (excluding first and last bit)
        crossover_point = np.random.randint(1, offspring_size[1] - 1)

        # Apply crossover only if a random number is below the rate
        if np.random.rand() < crossover_rate:
            # Create offspring
            offspring[k, 0:crossover_point] = parent1[0:crossover_point]
            offspring[k, crossover_point:] = parent2[crossover_point:]
        else:
            # No crossover, just copy Parent 1
            offspring[k, :] = parent1

    return offspring

def mutation(offspring_crossover, mutation_rate=0.01):
    """Flips a bit in the offspring based on a small mutation rate."""
    for idx in range(offspring_crossover.shape[0]):
        # Iterate through each gene (bit)
        for bit_idx in range(offspring_crossover.shape[1]):
            if np.random.rand() < mutation_rate:
                # Flip the bit (0 becomes 1, 1 becomes 0)
                offspring_crossover[idx, bit_idx] = 1 - offspring_crossover[idx, bit_idx]
    return offspring_crossover

# 3. Main GA Loop
def genetic_algorithm():
    # --- Parameters ---
    POP_SIZE = 10         # Number of chromosomes in the population
    GENE_LENGTH = 5       # Length of the binary string (x range 0 to 31)
    NUM_GENERATIONS = 50  # Number of iterations
    NUM_PARENTS = 4       # Number of parents selected for reproduction

    # 3.1. Initialization
    population = initialize_population(POP_SIZE, GENE_LENGTH)
    best_fitness_history = []

    # Calculate how many offspring to create
    num_offspring = POP_SIZE - NUM_PARENTS

    # 3.2. Evolution Loop
    for generation in range(NUM_GENERATIONS):
        # Calculate fitness for the entire population
        fitnesses = np.array([fitness_function(chromosome) for chromosome in population])

        # Store the best result from this generation
        best_fitness = np.max(fitnesses)
        best_x_decimal = binary_to_decimal(population[np.argmax(fitnesses)])
        best_fitness_history.append(best_fitness)

        # --- 3.3. Selection ---
        parents = selection(population, fitnesses, NUM_PARENTS)

        # --- 3.4. Crossover ---
        offspring_crossover = crossover(parents, offspring_size=(num_offspring, GENE_LENGTH))

        # --- 3.5. Mutation ---
        offspring_mutation = mutation(offspring_crossover)

        # --- 3.6. New Generation (Elitism: keep the best parent) ---
        # The new population consists of the selected parents and the new offspring
        population[0:NUM_PARENTS, :] = parents
        population[NUM_PARENTS:, :] = offspring_mutation

        # Print status (optional)
        if (generation + 1) % 10 == 0 or generation == 0:
            print(f"Gen {generation+1}: Best x = {best_x_decimal}, Max f(x) = {best_fitness}")

    # 4. Final Result
    final_fitnesses = np.array([fitness_function(chromosome) for chromosome in population])
    best_index = np.argmax(final_fitnesses)
    final_best_x = binary_to_decimal(population[best_index])
    final_best_f = final_fitnesses[best_index]

    print("\n" + "="*40)
    print(f"Final Optimal Solution (f(x) = x^2 + 5):")
    print(f"Optimal x: {final_best_x}")
    print(f"Maximum f(x): {final_best_f}")
    print(f"Binary Chromosome: {population[best_index]}")
    print("="*40)

if __name__ == "__main__":
    genetic_algorithm()

Gen 1: Best x = 23, Max f(x) = 534
Gen 10: Best x = 23, Max f(x) = 534
Gen 20: Best x = 23, Max f(x) = 534
Gen 30: Best x = 23, Max f(x) = 534
Gen 40: Best x = 23, Max f(x) = 534
Gen 50: Best x = 31, Max f(x) = 966

Final Optimal Solution (f(x) = x^2 + 5):
Optimal x: 23
Maximum f(x): 534
Binary Chromosome: [1 0 1 1 1]
